> **Superseded (2026-09-22).** Use `python -m src.experiments.run_ablation` (Holm-corrected p-values and rank-biserial effect sizes are in `reports/ablation_results/significance_results.csv`). Kept for reference; its original outputs are in `archive/results_pre_fix_2026-09-22/`.

In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.multitest import multipletests

RESULTS_DIR = Path("../reports/ablation_results")   # adjust path to wherever you're running from

DATASETS = ["creditcard", "mammography"]
VARIANTS = ["ars_stack", "plain_smote", "unweighted_stacking"]

all_fold_results = {}
for dataset_name in DATASETS:
    for variant in VARIANTS:
        path = RESULTS_DIR / f"{dataset_name}_{variant}_folds.json"
        with open(path) as f:
            all_fold_results[(dataset_name, variant)] = json.load(f)

with open(RESULTS_DIR / "significance_results.json") as f:
    significance_results = json.load(f)

print("Loaded, no re-run needed. Keys available:", list(all_fold_results.keys()))

Loaded, no re-run needed. Keys available: [('creditcard', 'ars_stack'), ('creditcard', 'plain_smote'), ('creditcard', 'unweighted_stacking'), ('mammography', 'ars_stack'), ('mammography', 'plain_smote'), ('mammography', 'unweighted_stacking')]


In [4]:
def rank_biserial(scores_a, scores_b):
    diffs = np.array(scores_b) - np.array(scores_a)
    diffs = diffs[diffs != 0]
    ranks = pd.Series(np.abs(diffs)).rank().values
    w_pos = ranks[diffs > 0].sum()
    w_neg = ranks[diffs < 0].sum()
    return (w_pos - w_neg) / (w_pos + w_neg)

def holm_correct(comparison_dict, metrics=["f1", "auc_pr", "g_mean", "mcc"]):
    pvals = [comparison_dict[m]["p_value"] for m in metrics]
    reject, p_corrected, _, _ = multipletests(pvals, alpha=0.05, method="holm")
    return {m: {"reject": bool(r), "p_corrected": float(p)} for m, r, p in zip(metrics, reject, p_corrected)}

rows = []
for dataset_name in DATASETS:
    for comparison_name, (variant_a, variant_b) in [
        ("adaptive_resampling", ("plain_smote", "ars_stack")),
        ("confidence_weighting", ("unweighted_stacking", "ars_stack")),
    ]:
        comp = significance_results[dataset_name][
            "adaptive_resampling_effect" if comparison_name == "adaptive_resampling" else "confidence_weighting_effect"
        ]
        holm = holm_correct(comp)
        for metric in ["f1", "auc_pr", "g_mean", "mcc"]:
            scores_a = [f[metric] for f in all_fold_results[(dataset_name, variant_a)]]
            scores_b = [f[metric] for f in all_fold_results[(dataset_name, variant_b)]]
            r = rank_biserial(scores_a, scores_b)
            rows.append({
                "dataset": dataset_name, "comparison": comparison_name, "metric": metric,
                "p_raw": comp[metric]["p_value"], "p_holm": holm[metric]["p_corrected"],
                "significant_after_correction": holm[metric]["reject"],
                "effect_size_r": round(r, 3),
            })

results_table = pd.DataFrame(rows)
results_table.to_csv(RESULTS_DIR / "effect_sizes_corrected.csv", index=False)
print(results_table.to_string(index=False))

    dataset           comparison metric    p_raw   p_holm  significant_after_correction  effect_size_r
 creditcard  adaptive_resampling     f1 0.005371 0.010742                          True          0.783
 creditcard  adaptive_resampling auc_pr 0.719727 0.719727                         False          0.117
 creditcard  adaptive_resampling g_mean 0.000061 0.000244                          True         -1.000
 creditcard  adaptive_resampling    mcc 0.001160 0.003479                          True          0.883
 creditcard confidence_weighting     f1 0.005772 0.017315                          True          0.868
 creditcard confidence_weighting auc_pr 0.000061 0.000244                          True          1.000
 creditcard confidence_weighting g_mean 0.005772 0.017315                          True          0.868
 creditcard confidence_weighting    mcc 0.008775 0.017315                          True          0.824
mammography  adaptive_resampling     f1 0.488708 1.000000                